In [1]:
import copy
import json
import os
import random
from pathlib import Path
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import pyreadstat
import sklearn
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

import flwr as fl
from flwr.simulation import start_simulation

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from scipy.spatial.distance import cdist
from datetime import datetime, timezone

## Load and audit the original NDHS data

In [2]:
# ------------------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------------------

DATA_PATH = Path(r"c:\Users\usher\Downloads\NGIR7BDT\NGIR7BFL.DTA")

OUTPUT_DIR = Path("NDHS_FL_Research")
AUDIT_DIR = OUTPUT_DIR / "01_data_audit"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. LOAD ORIGINAL DATA
# ------------------------------------------------------------

df, meta = pyreadstat.read_dta(
    str(DATA_PATH),
    apply_value_formats=False
)

print("DATASET LOADED")
print("=" * 60)
print("Dataset shape:", df.shape)
print("Number of observations:", len(df))
print("Number of variables:", len(df.columns))

# ------------------------------------------------------------
# 3. REQUIRED VARIABLE CHECK
# ------------------------------------------------------------

required_variables = [
    "m15_1",
    "v024",
    "v001",
    "v021",
    "v005",
    "v022",
    "v023",
    "v501",
    "v025",
    "v106",
    "v190",
    "v714",
    "bord_01",
    "m14_1",
    "v467d",
    "v157",
    "v158",
    "v159",
    "v701",
    "v130",
    "v131",
    "b3_01",
    "v011"
]

missing_variables = [
    col for col in required_variables
    if col not in df.columns
]

if missing_variables:
    raise ValueError(
        f"Required variables missing from dataset: {missing_variables}"
    )

print("\nAll required variables are present.")

# ------------------------------------------------------------
# 4. RECONSTRUCT THE BINARY OUTCOME
# ------------------------------------------------------------

facility_codes = [21, 22, 23, 26, 31, 36]
home_codes = [11, 12]

df["m15_1"] = pd.to_numeric(df["m15_1"], errors="coerce")

df["facility_delivery"] = np.select(
    [
        df["m15_1"].isin(home_codes),
        df["m15_1"].isin(facility_codes)
    ],
    [
        0,
        1
    ],
    default=np.nan
)

# Keep only observations with a valid binary outcome.
model_df = df.loc[
    df["facility_delivery"].notna()
].copy()

model_df["facility_delivery"] = (
    model_df["facility_delivery"].astype(int)
)

print("\nOUTCOME RECONSTRUCTION")
print("=" * 60)

print("Original observations:", len(df))
print("Eligible observations:", len(model_df))
print("Excluded observations:", len(df) - len(model_df))

print("\nOutcome distribution:")
print(
    model_df["facility_delivery"]
    .value_counts()
    .sort_index()
)

print("\nOutcome percentages:")
print(
    model_df["facility_delivery"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

# ------------------------------------------------------------
# 5. CHECK REGIONAL DISTRIBUTION
# ------------------------------------------------------------

region_summary = (
    model_df.groupby("v024")
    .agg(
        sample_size=("facility_delivery", "size"),
        facility_deliveries=("facility_delivery", "sum"),
        facility_prevalence=("facility_delivery", "mean")
    )
    .reset_index()
)

region_summary["facility_prevalence"] *= 100

print("\nREGIONAL DISTRIBUTION")
print("=" * 60)
print(region_summary.to_string(index=False))

region_summary.to_csv(
    AUDIT_DIR / "regional_outcome_distribution.csv",
    index=False
)

# ------------------------------------------------------------
# 6. VERIFY PSU AND CLUSTER IDENTIFIERS
# ------------------------------------------------------------

print("\nPSU AND CLUSTER CHECKS")
print("=" * 60)

print("v001 missing:", df["v001"].isna().sum())
print("v021 missing:", df["v021"].isna().sum())

same_cluster = (
    df["v001"].astype("string")
    == df["v021"].astype("string")
)

print("v001 equals v021 for all nonmissing pairs:",
      same_cluster[df["v001"].notna() & df["v021"].notna()].all())

print("Unique v001:", df["v001"].nunique())
print("Unique v021:", df["v021"].nunique())

print(
    "Unique PSUs in eligible sample:",
    model_df["v021"].nunique()
)

# Check whether a PSU is assigned to more than one region.
psu_zone_counts = (
    model_df.groupby("v021")["v024"]
    .nunique()
)

print(
    "PSUs appearing in multiple regions:",
    int((psu_zone_counts > 1).sum())
)

# ------------------------------------------------------------
# 7. SURVEY WEIGHT AND STRATA CHECKS
# ------------------------------------------------------------

model_df["weight"] = (
    pd.to_numeric(model_df["v005"], errors="coerce") / 1_000_000
)

print("\nSURVEY DESIGN CHECKS")
print("=" * 60)

print("Missing weights:", model_df["weight"].isna().sum())
print("Non-positive weights:", (model_df["weight"] <= 0).sum())
print("Unique PSUs:", model_df["v021"].nunique())
print("Unique v022 strata:", model_df["v022"].nunique())
print("Unique v023 values:", model_df["v023"].nunique())

if model_df["weight"].isna().any():
    raise ValueError("Missing survey weights found in eligible sample.")

if (model_df["weight"] <= 0).any():
    raise ValueError("Non-positive survey weights found.")

# Weighted and unweighted outcome prevalence.
unweighted_prevalence = model_df["facility_delivery"].mean()

weighted_prevalence = np.average(
    model_df["facility_delivery"],
    weights=model_df["weight"]
)

print("\nUnweighted facility prevalence:",
      round(unweighted_prevalence * 100, 3), "%")

print("Weighted facility prevalence:",
      round(weighted_prevalence * 100, 3), "%")

# Regional weighted prevalence.
weighted_region_rows = []

for zone, group in model_df.groupby("v024"):

    weighted_rate = np.average(
        group["facility_delivery"],
        weights=group["weight"]
    )

    weighted_region_rows.append({
        "v024": zone,
        "sample_size": len(group),
        "unweighted_prevalence": group["facility_delivery"].mean(),
        "weighted_prevalence": weighted_rate
    })

weighted_region_summary = pd.DataFrame(weighted_region_rows)

weighted_region_summary[
    ["unweighted_prevalence", "weighted_prevalence"]
] *= 100

weighted_region_summary.to_csv(
    AUDIT_DIR / "weighted_regional_prevalence.csv",
    index=False
)

print("\nWeighted regional prevalence:")
print(weighted_region_summary.to_string(index=False))

# ------------------------------------------------------------
# 8. INSPECT v701 VALUE LABELS AND MISSINGNESS
# ------------------------------------------------------------

print("\nPARTNER EDUCATION: v701")
print("=" * 60)

print("Variable label:")
print(meta.column_names_to_labels.get("v701", "No label available"))

print("\nValue labels:")
print(meta.variable_value_labels.get("v701", "No value labels available"))

print("\nObserved v701 frequencies, including missing:")
print(
    model_df["v701"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nMissing v701 by marital status (v501):")

v701_missing_by_marital = (
    model_df.assign(v701_missing=model_df["v701"].isna())
    .groupby("v501", dropna=False)
    .agg(
        sample_size=("v701_missing", "size"),
        missing_v701=("v701_missing", "sum")
    )
    .reset_index()
)

v701_missing_by_marital["missing_percentage"] = (
    100 * v701_missing_by_marital["missing_v701"]
    / v701_missing_by_marital["sample_size"]
)

print(v701_missing_by_marital.to_string(index=False))

v701_missing_by_marital.to_csv(
    AUDIT_DIR / "v701_missingness_by_marital_status.csv",
    index=False
)

# ------------------------------------------------------------
# 9. FEATURE MISSINGNESS AUDIT
# ------------------------------------------------------------

baseline_features = [
    "age_at_birth",
    "v025",
    "v106",
    "v190",
    "v714",
    "bord_01"
]

expanded_features = baseline_features + [
    "m14_1",
    "v467d",
    "v157",
    "v158",
    "v159",
    "v701",
    "v130",
    "v131"
]

# Derive age at birth.
model_df["age_at_birth"] = np.floor(
    (pd.to_numeric(model_df["b3_01"], errors="coerce")
     - pd.to_numeric(model_df["v011"], errors="coerce")) / 12
)

feature_missingness = pd.DataFrame({
    "feature": expanded_features,
    "missing_n": [
        model_df[col].isna().sum()
        for col in expanded_features
    ],
    "missing_percentage": [
        model_df[col].isna().mean() * 100
        for col in expanded_features
    ]
})

print("\nFEATURE MISSINGNESS")
print("=" * 60)
print(feature_missingness.to_string(index=False))

feature_missingness.to_csv(
    AUDIT_DIR / "feature_missingness.csv",
    index=False
)

# ------------------------------------------------------------
# 10. SAVE AUDIT SUMMARY
# ------------------------------------------------------------

audit_summary = {
    "source_file": DATA_PATH.name,
    "original_rows": int(len(df)),
    "original_columns": int(len(df.columns)),
    "eligible_rows": int(len(model_df)),
    "home_deliveries": int(
        (model_df["facility_delivery"] == 0).sum()
    ),
    "facility_deliveries": int(
        (model_df["facility_delivery"] == 1).sum()
    ),
    "unweighted_facility_prevalence": float(
        unweighted_prevalence
    ),
    "weighted_facility_prevalence": float(
        weighted_prevalence
    ),
    "unique_psus": int(model_df["v021"].nunique()),
    "unique_v022_strata": int(model_df["v022"].nunique()),
    "v001_equals_v021": bool(
        same_cluster[
            df["v001"].notna() & df["v021"].notna()
        ].all()
    )
}

with open(
    AUDIT_DIR / "audit_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(audit_summary, f, indent=4)

print("\nSTEP 1 COMPLETED")
print("Audit outputs saved to:", AUDIT_DIR.resolve())

DATASET LOADED
Dataset shape: (41821, 5394)
Number of observations: 41821
Number of variables: 5394

All required variables are present.

OUTCOME RECONSTRUCTION
Original observations: 41821
Eligible observations: 21442
Excluded observations: 20379

Outcome distribution:
facility_delivery
0    12441
1     9001
Name: count, dtype: int64

Outcome percentages:
facility_delivery
0    58.02
1    41.98
Name: proportion, dtype: float64

REGIONAL DISTRIBUTION
 v024  sample_size  facility_deliveries  facility_prevalence
    1         3838                 1997            52.032308
    2         4498                 1169            25.989329
    3         6307                  989            15.680989
    4         2338                 1887            80.710009
    5         2050                  998            48.682927
    6         2411                 1961            81.335545

PSU AND CLUSTER CHECKS
v001 missing: 0
v021 missing: 0
v001 equals v021 for all nonmissing pairs: True
Unique v001: 1

## Construct the clean analytical dataset

In [3]:
DATA_PATH = Path(r"c:\Users\usher\Downloads\NGIR7BDT\NGIR7BFL.DTA")

OUTPUT_DIR = Path("NDHS_FL_Research")
DATA_DIR = OUTPUT_DIR / "02_clean_datasets"

DATA_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. LOAD SOURCE DATA
# ------------------------------------------------------------

df,meta = pyreadstat.read_dta(
    str(DATA_PATH),
    apply_value_formats=False
)

# Preserve original row position for traceability.
df = df.copy()
df["source_row_id"] = df.index.astype(int)

# ------------------------------------------------------------
# 3. RECONSTRUCT OUTCOME
# ------------------------------------------------------------

facility_codes = [21, 22, 23, 26, 31, 36]
home_codes = [11, 12]

df["m15_1"] = pd.to_numeric(
    df["m15_1"],
    errors="coerce"
)

df["facility_delivery"] = np.select(
    [
        df["m15_1"].isin(home_codes),
        df["m15_1"].isin(facility_codes)
    ],
    [
        0,
        1
    ],
    default=np.nan
)

# Retain only observations with a valid outcome.
analytic_df = df.loc[
    df["facility_delivery"].notna()
].copy()

analytic_df["facility_delivery"] = (
    analytic_df["facility_delivery"].astype(int)
)

# ------------------------------------------------------------
# 4. DERIVE AGE AT BIRTH
# ------------------------------------------------------------

analytic_df["b3_01"] = pd.to_numeric(
    analytic_df["b3_01"],
    errors="coerce"
)

analytic_df["v011"] = pd.to_numeric(
    analytic_df["v011"],
    errors="coerce"
)

analytic_df["age_at_birth"] = np.floor(
    (analytic_df["b3_01"] - analytic_df["v011"]) / 12
)

# Check for invalid derived ages.
invalid_age = (
    analytic_df["age_at_birth"].isna()
    | (analytic_df["age_at_birth"] < 10)
    | (analytic_df["age_at_birth"] > 55)
)

print("Missing or out-of-range age_at_birth:",
      int(invalid_age.sum()))

# Do not automatically delete these records here.
# We will decide the appropriate treatment during preprocessing.

# ------------------------------------------------------------
# 5. DEFINE THE TWO CO-PRIMARY FEATURE CONFIGURATIONS
# ------------------------------------------------------------

baseline_features = [
    "age_at_birth",
    "v025",
    "v106",
    "v190",
    "v714",
    "bord_01"
]

expanded_features = baseline_features + [
    "m14_1",
    "v467d",
    "v157",
    "v158",
    "v159",
    "v701",
    "v130",
    "v131"
]

# ------------------------------------------------------------
# 6. CHECK REQUIRED VARIABLES
# ------------------------------------------------------------

required = (
    baseline_features
    + [
        "m14_1", "v467d", "v157", "v158",
        "v159", "v701", "v130", "v131",
        "v024", "v001", "v021",
        "v005", "v022", "v023",
        "v501", "facility_delivery",
        "source_row_id"
    ]
)

missing = [
    col for col in required
    if col not in analytic_df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

# ------------------------------------------------------------
# 7. CREATE A PROVISIONAL CATEGORICAL VERSION OF v701
# ------------------------------------------------------------

# Preserve the original numeric v701 values.
analytic_df["v701_raw"] = analytic_df["v701"]

# Retrieve DHS value labels, if present.
v701_labels = meta.variable_value_labels.get("v701", {})

# Convert known numeric codes to readable categories.
def label_v701(value):

    if pd.isna(value):
        return "Missing - classification pending"

    numeric_value = int(value)

    if numeric_value in v701_labels:
        return str(v701_labels[numeric_value])

    return f"Code {numeric_value}"


analytic_df["v701_category"] = (
    analytic_df["v701_raw"].apply(label_v701)
)

print("\nv701 category distribution:")
print(
    analytic_df["v701_category"]
    .value_counts(dropna=False)
)

# NOTE:
# "Missing - classification pending" is a provisional category.
# Do not regard it as a confirmed DHS response category.
# Review Step 1's v701 labels and marital-status cross-tabulation
# before using this variable in final model preprocessing.

# ------------------------------------------------------------
# 8. CREATE CORE ANALYTICAL DATASET
# ------------------------------------------------------------

core_columns = [
    "source_row_id",
    "facility_delivery",
    "v024"
]

core_df = analytic_df[core_columns].copy()

# ------------------------------------------------------------
# 9. CREATE BASELINE FEATURE DATASET
# ------------------------------------------------------------

baseline_columns = (
    ["source_row_id"]
    + baseline_features
    + ["facility_delivery", "v024"]
)

baseline_df = analytic_df[baseline_columns].copy()

# ------------------------------------------------------------
# 10. CREATE EXPANDED FEATURE DATASET
# ------------------------------------------------------------

expanded_columns = (
    ["source_row_id"]
    + baseline_features
    + [
        "m14_1",
        "v467d",
        "v157",
        "v158",
        "v159",
        "v130",
        "v131",
        "facility_delivery",
        "v024"
    ]
)

expanded_df = analytic_df[expanded_columns].copy()

# Use the readable provisional category for v701.
expanded_df["v701"] = analytic_df["v701_category"]

# ------------------------------------------------------------
# 11. CREATE SEPARATE SURVEY-DESIGN METADATA
# ------------------------------------------------------------

metadata_columns = [
    "source_row_id",
    "v024",
    "v001",
    "v021",
    "v005",
    "v022",
    "v023",
    "v501"
]

survey_metadata = analytic_df[metadata_columns].copy()

survey_metadata["survey_weight"] = (
    pd.to_numeric(
        survey_metadata["v005"],
        errors="coerce"
    ) / 1_000_000
)

# ------------------------------------------------------------
# 12. INTEGRITY CHECKS
# ------------------------------------------------------------

assert len(core_df) == len(baseline_df)
assert len(core_df) == len(expanded_df)
assert len(core_df) == len(survey_metadata)

assert core_df["source_row_id"].is_unique
assert baseline_df["source_row_id"].is_unique
assert expanded_df["source_row_id"].is_unique
assert survey_metadata["source_row_id"].is_unique

assert (
    core_df["source_row_id"].values
    == baseline_df["source_row_id"].values
).all()

assert (
    core_df["source_row_id"].values
    == expanded_df["source_row_id"].values
).all()

assert (
    core_df["source_row_id"].values
    == survey_metadata["source_row_id"].values
).all()

assert set(core_df["facility_delivery"].unique()) == {0, 1}

assert set(core_df["v024"].dropna().unique()) == {
    1, 2, 3, 4, 5, 6
}

assert survey_metadata["survey_weight"].notna().all()
assert (survey_metadata["survey_weight"] > 0).all()

# ------------------------------------------------------------
# 13. REPORT DATASET DIMENSIONS
# ------------------------------------------------------------

print("\nCLEAN DATASET SUMMARY")
print("=" * 60)

print("Analytical observations:", len(core_df))

print(
    "Baseline source predictors:",
    len(baseline_features)
)

print(
    "Expanded source predictors:",
    len(expanded_features)
)

print("Baseline dataset shape:", baseline_df.shape)
print("Expanded dataset shape:", expanded_df.shape)
print("Survey metadata shape:", survey_metadata.shape)

print("\nBaseline missingness:")
print(
    baseline_df[baseline_features]
    .isna()
    .sum()
)

print("\nExpanded missingness:")
print(
    expanded_df[expanded_features]
    .isna()
    .sum()
)

# ------------------------------------------------------------
# 14. SAVE DATASETS
# ------------------------------------------------------------

core_df.to_csv(
    DATA_DIR / "analytic_core.csv",
    index=False
)

baseline_df.to_csv(
    DATA_DIR / "baseline_features.csv",
    index=False
)

expanded_df.to_csv(
    DATA_DIR / "expanded_features.csv",
    index=False
)

survey_metadata.to_csv(
    DATA_DIR / "survey_design_metadata.csv",
    index=False
)

# ------------------------------------------------------------
# 15. SAVE FEATURE CONFIGURATION AND DATA MANIFEST
# ------------------------------------------------------------

manifest = {
    "study": (
        "Federated Learning Under Geographic Structural "
        "Heterogeneity Across Nigeria's Six Geopolitical Zones"
    ),
    "source_file": DATA_PATH.name,
    "analytical_sample_size": int(len(core_df)),
    "target": "facility_delivery",
    "client_assignment": "v024",
    "baseline_features": baseline_features,
    "expanded_features": expanded_features,
    "survey_metadata": [
        "v005",
        "v021",
        "v022",
        "v023",
        "v001"
    ],
    "weight_formula": "v005 / 1000000",
    "v701_status": (
        "Provisional categorical representation; "
        "missingness classification requires review."
    ),
    "preprocessing_performed": False,
    "train_test_split_performed": False,
    "model_training_performed": False
}

with open(
    DATA_DIR / "dataset_manifest.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(manifest, f, indent=4)

print("\nSTEP 2 COMPLETED")
print("Datasets saved to:", DATA_DIR.resolve())
print("\nNo train/test split or model training has been performed.")


Missing or out-of-range age_at_birth: 0

v701 category distribution:
v701_category
no education                        7121
secondary                           6857
higher                              2998
primary                             2848
Missing - classification pending    1339
don't know                           279
Name: count, dtype: int64

CLEAN DATASET SUMMARY
Analytical observations: 21442
Baseline source predictors: 6
Expanded source predictors: 14
Baseline dataset shape: (21442, 9)
Expanded dataset shape: (21442, 17)
Survey metadata shape: (21442, 9)

Baseline missingness:
age_at_birth    0
v025            0
v106            0
v190            0
v714            0
bord_01         0
dtype: int64

Expanded missingness:
age_at_birth    0
v025            0
v106            0
v190            0
v714            0
bord_01         0
m14_1           0
v467d           0
v157            0
v158            0
v159            0
v701            0
v130            0
v131            0
dtype:

In [4]:
## additional diagnostic checks for v701 missingness by marital status

# Reconstruct eligible analytical sample
facility_codes = [21, 22, 23, 26, 31, 36]
home_codes = [11, 12]

df["facility_delivery"] = np.select(
    [
        df["m15_1"].isin(home_codes),
        df["m15_1"].isin(facility_codes)
    ],
    [0, 1],
    default=np.nan
)

analytic_df = df.loc[
    df["facility_delivery"].notna()
].copy()

# Identify missing v701
analytic_df["v701_missing"] = (
    analytic_df["v701"].isna()
)

# Inspect marital status labels
print("v501 variable label:")
print(meta.column_names_to_labels.get("v501"))

print("\nv501 value labels:")
print(meta.variable_value_labels.get("v501"))

print("\nv701 variable label:")
print(meta.column_names_to_labels.get("v701"))

print("\nv701 value labels:")
print(meta.variable_value_labels.get("v701"))

# Cross-tabulation
cross_tab = pd.crosstab(
    analytic_df["v501"],
    analytic_df["v701_missing"],
    margins=True
)

print("\nMissing v701 by marital status:")
print(cross_tab)

# Percent missing within each marital status
missing_rate = (
    analytic_df.groupby("v501")["v701_missing"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

missing_rate["missing_percentage"] = (
    missing_rate["mean"] * 100
)

print("\nMissingness percentages:")
print(missing_rate.to_string(index=False))

# Save diagnostic outputs
cross_tab.to_csv("v701_marital_status_crosstab.csv")
missing_rate.to_csv(
    "v701_marital_status_missingness.csv",
    index=False
)

v501 variable label:
current marital status

v501 value labels:
{0: 'never in union', 1: 'married', 2: 'living with partner', 3: 'widowed', 4: 'divorced', 5: 'no longer living together/separated'}

v701 variable label:
husband/partner's education level

v701 value labels:
{0: 'no education', 1: 'primary', 2: 'secondary', 3: 'higher', 8: "don't know"}

Missing v701 by marital status:
v701_missing  False  True    All
v501                            
0                 0   583    583
1             19435     0  19435
2               668     0    668
3                 0   268    268
4                 0   245    245
5                 0   243    243
All           20103  1339  21442

Missingness percentages:
 v501  count  sum  mean  missing_percentage
    0    583  583   1.0               100.0
    1  19435    0   0.0                 0.0
    2    668    0   0.0                 0.0
    3    268  268   1.0               100.0
    4    245  245   1.0               100.0
    5    243  243   1.0    

In [5]:
ROOT = Path(
    r"C:\Users\usher\Documents\NDHS-Fl-Research\NDHS_FL_Research"
)
CLEAN = ROOT / "02_clean_datasets"

expanded_path = CLEAN / "expanded_features.csv"
metadata_path = CLEAN / "survey_design_metadata.csv"

expanded = pd.read_csv(expanded_path)
metadata = pd.read_csv(metadata_path)

# Confirm that the records align using the source row identifier.
assert "source_row_id" in expanded.columns
assert "source_row_id" in metadata.columns
assert expanded["source_row_id"].is_unique
assert metadata["source_row_id"].is_unique

df = expanded.merge(
    metadata[["source_row_id", "v501"]],
    on="source_row_id",
    how="left",
    validate="one_to_one"
)

assert df["v501"].notna().all()

# Identify provisional missing labels and any remaining nulls.
missing_labels = {
    "Missing - classification pending",
    "Missing",
    "NaN"
}

is_missing_v701 = (
    df["v701"].isna()
    | df["v701"].astype(str).isin(missing_labels)
)

# Current union: v501 = 1 married, 2 living with partner.
current_union = df["v501"].isin([1, 2])

# Check the observed relationship before recoding.
assert not (is_missing_v701 & current_union).any(), (
    "Unexpected missing v701 among respondents currently in a union."
)
assert not ((~is_missing_v701) & (~current_union)).any(), (
    "Unexpected observed v701 among respondents not currently in a union."
)

df.loc[is_missing_v701, "v701"] = (
    "not applicable - no current union"
)

# Keep don't know as its own category.
assert (df["v701"] == "don't know").sum() == 279

# Save without the temporary marital-status join column.
df = df.drop(columns=["v501"])

df.to_csv(expanded_path, index=False)

print("UPDATED v701 DISTRIBUTION")
print(df["v701"].value_counts(dropna=False))

print("\nRemaining missing v701:", df["v701"].isna().sum())
print("Expanded dataset shape:", df.shape)

assert df["v701"].notna().all()
assert len(df) == 21442

decision = {
    "variable": "v701",
    "treatment": (
        "Categorical predictor; retain observed education categories, "
        "'don't know', and a distinct structural non-applicability category."
    ),
    "structural_missingness": (
        "Missing for respondents not currently in a union "
        "(v501 = 0, 3, 4, 5)."
    ),
    "observed_categories": [
        "no education",
        "primary",
        "secondary",
        "higher",
        "don't know",
        "not applicable - no current union"
    ],
    "imputation": "None at this stage",
    "encoding": "To be fitted using training data only"
}

with open(CLEAN / "v701_treatment_decision.json", "w") as f:
    json.dump(decision, f, indent=4)

print("\nDecision saved.")

UPDATED v701 DISTRIBUTION
v701
no education                         7121
secondary                            6857
higher                               2998
primary                              2848
not applicable - no current union    1339
don't know                            279
Name: count, dtype: int64

Remaining missing v701: 0
Expanded dataset shape: (21442, 17)

Decision saved.


## Creating the fixed geographic, PSU-level split (Primary Sampling Unit)

In [6]:
# --------------------------------------------------
# 1. Paths and configuration
# --------------------------------------------------

ROOT = Path(
    r"C:\Users\usher\Documents\NDHS-Fl-Research\NDHS_FL_Research"
)

CLEAN = ROOT / "02_clean_datasets"
SPLIT_DIR = ROOT / "03_partitions"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_PATH = CLEAN / "baseline_features.csv"
EXPANDED_PATH = CLEAN / "expanded_features.csv"
METADATA_PATH = CLEAN / "survey_design_metadata.csv"

SPLIT_SEED = 2026
N_SPLITS = 5

# --------------------------------------------------
# 2. Load datasets
# --------------------------------------------------

baseline = pd.read_csv(BASELINE_PATH)
expanded = pd.read_csv(EXPANDED_PATH)
metadata = pd.read_csv(METADATA_PATH)

required_metadata = [
    "source_row_id", "v024", "v021", "v005",
    "v022", "v023", "survey_weight"
]

for col in required_metadata:
    assert col in metadata.columns, f"Missing metadata column: {col}"

for name, data in [
    ("baseline", baseline),
    ("expanded", expanded),
    ("metadata", metadata)
]:
    assert "source_row_id" in data.columns
    assert data["source_row_id"].is_unique, (
        f"Duplicate source_row_id in {name}"
    )

# Both feature configurations must contain exactly
# the same respondents.
baseline_ids = set(baseline["source_row_id"])
expanded_ids = set(expanded["source_row_id"])
metadata_ids = set(metadata["source_row_id"])

assert baseline_ids == expanded_ids
assert baseline_ids == metadata_ids

# Confirm target and regional assignment agree.
core = baseline[
    ["source_row_id", "facility_delivery", "v024"]
].copy()

expanded_check = expanded[
    ["source_row_id", "facility_delivery", "v024"]
].copy()

check = core.merge(
    expanded_check,
    on="source_row_id",
    suffixes=("_base", "_expanded"),
    validate="one_to_one"
)

assert (
    check["facility_delivery_base"]
    == check["facility_delivery_expanded"]
).all()

assert (
    check["v024_base"] == check["v024_expanded"]
).all()

# Attach the PSU and survey metadata.
# Firstmetadata that both sources agree on region
region_check = baseline[
                        ["source_row_id", "v024"]
                        ].merge(
                            metadata[["source_row_id", "v024"]],
                            on="source_row_id",
                            suffixes=("_features", "_metadata"),
                            validate="one_to_one"
                        )

assert(
    region_check["v024_features"] == region_check[ "v024_metadata"]
).all(), "Region mismatch between feature and meteadata files."

# Keep the region already in core
#Exclude v024 from the metadat columns being merged
metadata_columns = [
    "source_row_id",
    "v021",
    "v005",
    "v022",
    "v023",
    "survey_weight"
]

core = core.merge(
    metadata[metadata_columns],
    on="source_row_id",
    how="left",
    validate='one_to_one'
)

# confirm expected columns
print("CORE COLUMNS:")
print(core.columns.tolist())

assert "v024" in core.columns
assert "v021" in core.columns
assert core["v021"].notna().all()
assert core["v024"].notna().all()
assert core["facility_delivery"].isin([0, 1]).all()

# Each PSU must belong to exactly one region.
psu_region_counts = core.groupby("v021")["v024"].nunique()
assert (psu_region_counts == 1).all()

# --------------------------------------------------
# 3. Construct region-outcome stratification labels
# --------------------------------------------------

core["stratum"] = (
    core["v024"].astype(str)
    + "_"
    + core["facility_delivery"].astype(str)
)

# Five folds: approximately 20% each.
# PSU grouping prevents cluster leakage.
splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SPLIT_SEED
)

fold_assignment = np.full(len(core), -1, dtype=int)

for fold, (_, fold_index) in enumerate(
    splitter.split(
        X=np.zeros((len(core), 1)),
        y=core["stratum"],
        groups=core["v021"]
    )
):
    fold_assignment[fold_index] = fold

assert (fold_assignment >= 0).all()

# Fold 0 = test; fold 1 = validation;
# folds 2, 3, 4 = training.
core["partition"] = np.select(
    [
        fold_assignment == 0,
        fold_assignment == 1
    ],
    [
        "test",
        "validation"
    ],
    default="train"
)

# --------------------------------------------------
# 4. Integrity checks
# --------------------------------------------------

# No PSU may cross partitions.
psu_partition_counts = (
    core.groupby("v021")["partition"].nunique()
)

assert (psu_partition_counts == 1).all(), (
    "PSU leakage detected across partitions."
)

# All regions and both outcome classes must occur
# in every partition.
coverage = pd.crosstab(
    [core["partition"], core["v024"]],
    core["facility_delivery"]
).reindex(columns=[0, 1], fill_value=0)

print("\nREGION / OUTCOME PARTITION COVERAGE")
print(coverage)

assert (coverage > 0).all().all(), (
    "At least one region/outcome combination is absent "
    "from a partition. Do not proceed."
)

# --------------------------------------------------
# 5. Partition summaries
# --------------------------------------------------

summary = (
    core.groupby(["partition", "v024"])
    .agg(
        respondents=("source_row_id", "count"),
        home_deliveries=(
            "facility_delivery",
            lambda x: int((x == 0).sum())
        ),
        facility_deliveries=(
            "facility_delivery",
            lambda x: int((x == 1).sum())
        ),
        psus=("v021", "nunique"),
        weighted_population=("survey_weight", "sum")
    )
    .reset_index()
)

summary["facility_prevalence"] = (
    summary["facility_deliveries"]
    / summary["respondents"]
)

print("\nPARTITION SUMMARY")
print(summary.to_string(index=False))

print("\nPARTITION TOTALS")
print(core["partition"].value_counts())

print("\nUNIQUE PSUs")
print(core.groupby("partition")["v021"].nunique())

# --------------------------------------------------
# 6. Save a single authoritative partition assignment
# --------------------------------------------------

assignments = core[
    ["source_row_id", "v024", "v021", "facility_delivery",
     "partition"]
].copy()

assignments.to_csv(
    SPLIT_DIR / "fixed_partition_assignments.csv",
    index=False
)

# Apply identical respondent assignments to both datasets.
for name, data in [
    ("baseline", baseline),
    ("expanded", expanded)
]:
    output = data.merge(
        assignments[["source_row_id", "partition"]],
        on="source_row_id",
        how="left",
        validate="one_to_one"
    )

    assert output["partition"].notna().all()

    output.to_csv(
        SPLIT_DIR / f"{name}_partitioned.csv",
        index=False
    )

# --------------------------------------------------
# 7. Save split manifest
# --------------------------------------------------

manifest = {
    "dataset": "2018 Nigeria Demographic and Health Survey",
    "analytical_observations": int(len(core)),
    "split_method": "StratifiedGroupKFold",
    "stratification": "geopolitical zone x facility_delivery",
    "grouping_variable": "v021",
    "n_splits": N_SPLITS,
    "split_seed": SPLIT_SEED,
    "fold_assignment": {
        "test": 0,
        "validation": 1,
        "train": [2, 3, 4]
    },
    "intended_proportions": {
        "train": 0.60,
        "validation": 0.20,
        "test": 0.20
    },
    "feature_configurations_share_assignments": True,
    "psu_leakage_check_passed": True,
    "region_outcome_coverage_check_passed": True,
    "model_training_performed": False
}

with open(SPLIT_DIR / "partition_manifest.json", "w") as f:
    json.dump(manifest, f, indent=4)

summary.to_csv(
    SPLIT_DIR / "partition_summary_by_region.csv",
    index=False
)

print("\nSTEP 3 COMPLETED")
print("Saved to:", SPLIT_DIR)
print("No model training or preprocessing has been performed.")


CORE COLUMNS:
['source_row_id', 'facility_delivery', 'v024', 'v021', 'v005', 'v022', 'v023', 'survey_weight']

REGION / OUTCOME PARTITION COVERAGE
facility_delivery     0     1
partition  v024              
test       1        367   399
           2        665   232
           3       1075   196
           4         90   376
           5        211   200
           6         89   392
train      1       1105  1199
           2       1997   701
           3       3174   593
           4        271  1131
           5        632   599
           6        272  1177
validation 1        369   399
           2        667   236
           3       1069   200
           4         90   380
           5        209   199
           6         89   392

PARTITION SUMMARY
 partition  v024  respondents  home_deliveries  facility_deliveries  psus  weighted_population  facility_prevalence
      test     1          766              367                  399    49           691.422458             0.520888
  

## Auditing features before preprocessing

In [7]:
SPLIT_DIR = ROOT / "03_partitions"
AUDIT_DIR = ROOT / "04_preprocessing_audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

datasets = {
    "baseline": pd.read_csv(
        SPLIT_DIR / "baseline_partitioned.csv"
    ),
    "expanded": pd.read_csv(
        SPLIT_DIR / "expanded_partitioned.csv"
    )
}

baseline_features = [
    "age_at_birth", "v025", "v106",
    "v190", "v714", "bord_01"
]

expanded_features = baseline_features + [
    "m14_1", "v467d", "v157", "v158",
    "v159", "v701", "v130", "v131"
]

feature_sets = {
    "baseline": baseline_features,
    "expanded": expanded_features
}

# Variables that should be treated as categorical,
# even if represented by numeric codes.
categorical_features = [
    "v025", "v106", "v190", "v714", "bord_01",
    "v467d", "v157", "v158", "v159",
    "v701", "v130", "v131"
]

audit_records = []
category_records = []
partition_missingness = []

for name, df in datasets.items():

    features = feature_sets[name]

    print("\n" + "=" * 65)
    print(name.upper(), "FEATURE AUDIT")
    print("=" * 65)

    assert set(features).issubset(df.columns)
    assert df["source_row_id"].is_unique
    assert df["partition"].isin(
        ["train", "validation", "test"]
    ).all()

    for col in features:

        series = df[col]
        is_categorical = col in categorical_features

        record = {
            "configuration": name,
            "feature": col,
            "dtype": str(series.dtype),
            "measurement_type": (
                "categorical" if is_categorical else "continuous"
            ),
            "missing_total": int(series.isna().sum()),
            "unique_values": int(series.nunique(dropna=True))
        }

        if pd.api.types.is_numeric_dtype(series):
            record["minimum"] = (
                float(series.min()) if series.notna().any() else None
            )
            record["maximum"] = (
                float(series.max()) if series.notna().any() else None
            )
            record["mean"] = (
                float(series.mean()) if series.notna().any() else None
            )
        else:
            record["minimum"] = None
            record["maximum"] = None
            record["mean"] = None

        audit_records.append(record)

        print(f"\n{col}")
        print("dtype:", series.dtype)
        print("unique:", series.nunique(dropna=True))
        print("missing:", series.isna().sum())

        if is_categorical:
            counts = (
                series.astype("string")
                .fillna("<MISSING>")
                .value_counts(dropna=False)
            )

            print("category frequencies:")
            print(counts.head(30).to_string())

            for category, count in counts.items():
                category_records.append({
                    "configuration": name,
                    "feature": col,
                    "category": str(category),
                    "count": int(count)
                })

        else:
            print("numeric summary:")
            print(series.describe().to_string())

        # Missingness by partition.
        for partition in ["train", "validation", "test"]:
            subset = df.loc[df["partition"] == partition, col]

            partition_missingness.append({
                "configuration": name,
                "feature": col,
                "partition": partition,
                "count": int(len(subset)),
                "missing": int(subset.isna().sum()),
                "missing_percentage": (
                    float(subset.isna().mean() * 100)
                )
            })

# --------------------------------------------------
# Check category consistency across partitions
# --------------------------------------------------

unknown_category_records = []

for name, df in datasets.items():

    for col in feature_sets[name]:

        if col not in categorical_features:
            continue

        train_categories = set(
            df.loc[df["partition"] == "train", col]
            .dropna().astype(str).unique()
        )

        for partition in ["validation", "test"]:

            evaluation_categories = set(
                df.loc[df["partition"] == partition, col]
                .dropna().astype(str).unique()
            )

            unseen = evaluation_categories - train_categories

            unknown_category_records.append({
                "configuration": name,
                "feature": col,
                "partition": partition,
                "unseen_category_count": len(unseen),
                "unseen_categories": " | ".join(sorted(unseen))
            })

# --------------------------------------------------
# Save audit outputs
# --------------------------------------------------

audit_df = pd.DataFrame(audit_records)
category_df = pd.DataFrame(category_records)
missingness_df = pd.DataFrame(partition_missingness)
unseen_df = pd.DataFrame(unknown_category_records)

audit_df.to_csv(
    AUDIT_DIR / "feature_type_and_range_audit.csv",
    index=False
)

category_df.to_csv(
    AUDIT_DIR / "feature_category_frequencies.csv",
    index=False
)

missingness_df.to_csv(
    AUDIT_DIR / "feature_missingness_by_partition.csv",
    index=False
)

unseen_df.to_csv(
    AUDIT_DIR / "unseen_categories_audit.csv",
    index=False
)

print("\n" + "=" * 65)
print("MISSINGNESS BY PARTITION")
print("=" * 65)
print(missingness_df.to_string(index=False))

print("\n" + "=" * 65)
print("CATEGORIES IN VALIDATION/TEST NOT SEEN IN TRAIN")
print("=" * 65)
print(unseen_df.to_string(index=False))

print("\nSTEP 4 FEATURE AUDIT COMPLETED")
print("Saved to:", AUDIT_DIR)
print("No preprocessing or model training performed.")



BASELINE FEATURE AUDIT

age_at_birth
dtype: float64
unique: 38
missing: 0
numeric summary:
count    21442.000000
mean        27.898797
std          6.948854
min         12.000000
25%         23.000000
50%         27.000000
75%         33.000000
max         49.000000

v025
dtype: int64
unique: 2
missing: 0
category frequencies:
v025
2    13912
1     7530

v106
dtype: int64
unique: 4
missing: 0
category frequencies:
v106
0    9501
2    6850
1    3324
3    1767

v190
dtype: int64
unique: 5
missing: 0
category frequencies:
v190
1    4997
2    4865
3    4493
4    3924
5    3163

v714
dtype: int64
unique: 2
missing: 0
category frequencies:
v714
1    14525
0     6917

bord_01
dtype: int64
unique: 17
missing: 0
category frequencies:
bord_01
2     3702
1     3657
3     3263
4     2848
5     2308
6     1798
7     1358
8      998
9      626
10     429
11     248
12     127
13      50
14      18
15       6
16       5
17       1

EXPANDED FEATURE AUDIT

age_at_birth
dtype: float64
unique: 38
missi

### verifying DHS special codes

In [8]:
SOURCE = Path(r"C:\Users\usher\Downloads\NGIR7BDT\NGIR7BFL.DTA")

# Read original DHS file and metadata.
raw, meta = pyreadstat.read_dta(SOURCE)

# Reconstruct the analytical sample using the established outcome rule.
eligible_codes = [11, 12, 21, 22, 23, 26, 31, 36]

analytic = raw.loc[
    raw["m15_1"].isin(eligible_codes)
].copy()

# Exclude code 96 (Other), as in the established sample.
analytic = analytic.loc[
    analytic["m15_1"] != 96
].copy()

variables = [
    "m14_1", "v130", "v131", "v190",
    "v025", "v106", "v714", "v467d",
    "v157", "v158", "v159", "v701",
    "bord_01"
]

for var in variables:
    print("\n" + "=" * 60)
    print("VARIABLE:", var)

    print("Variable label:")
    print(meta.column_names_to_labels.get(var, "No label available"))

    print("\nOriginal value labels:")
    print(meta.variable_value_labels.get(var, "No value labels available"))

    if var in analytic.columns:
        print("\nObserved values in analytical sample:")
        print(
            analytic[var]
            .value_counts(dropna=False)
            .sort_index()
            .to_string()
        )

print("\nDIAGNOSTIC COMPLETED")



VARIABLE: m14_1
Variable label:
number of antenatal visits during pregnancy

Original value labels:
{0: 'no antenatal visits', 98: "don't know"}

Observed values in analytical sample:
m14_1
0     5307
1      578
2      954
3     2217
4     2688
5     2309
6     2049
7     1041
8     1049
9      436
10     784
11     156
12     461
13      79
14     111
15     215
16     162
17      56
18     111
19      33
20     332
98     314

VARIABLE: v130
Variable label:
religion

Original value labels:
{1: 'catholic', 2: 'other christian', 3: 'islam', 4: 'traditionalist', 96: 'other'}

Observed values in analytical sample:
v130
1      1873
2      6770
3     12625
4        72
96      102

VARIABLE: v131
Variable label:
ethnicity

Original value labels:
{1: 'ekoi', 2: 'fulani', 3: 'hausa', 4: 'ibibio', 5: 'igala', 6: 'igbo', 7: 'ijaw/izon', 8: 'kanuri/beriberi', 9: 'tiv', 10: 'yoruba', 96: 'other', 98: "don't know"}

Observed values in analytical sample:
v131
1      108
2     1981
3     6754
4    

## fitting and validating the preprocessing pipelines

In [9]:
# ============================================================
# 1. PATHS AND CONFIGURATION
# ============================================================

ROOT = Path(
    r"C:\Users\usher\Documents\NDHS-Fl-Research\NDHS_FL_Research"
)

SPLIT_DIR = ROOT / "03_partitions"
OUTPUT_DIR = ROOT / "05_preprocessed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PARTITIONS = ["train", "validation", "test"]

SEED = 2026

# ============================================================
# 2. FEATURE DEFINITIONS
# ============================================================

BASELINE_CATEGORICAL = [
    "v025", "v106", "v190", "v714"
]

BASELINE_NUMERIC = [
    "age_at_birth", "bord_01"
]

EXPANDED_CATEGORICAL = BASELINE_CATEGORICAL + [
    "v467d", "v157", "v158", "v159",
    "v701", "v130", "v131"
]

# m14_1 is handled separately because code 98 means
# don't know, not 98 antenatal visits.
EXPANDED_NUMERIC = [
    "age_at_birth", "bord_01", "m14_1"
]

TARGET = "facility_delivery"
METADATA_COLUMNS = [
    "source_row_id", "v024", "partition"
]

# ============================================================
# 3. LOAD PARTITIONED DATASETS
# ============================================================

baseline = pd.read_csv(
    SPLIT_DIR / "baseline_partitioned.csv"
)

expanded = pd.read_csv(
    SPLIT_DIR / "expanded_partitioned.csv"
)

datasets = {
    "baseline": baseline,
    "expanded": expanded
}

# Verify common respondents and fixed assignments.
for name, df in datasets.items():

    required = (
        METADATA_COLUMNS
        + [TARGET]
    )

    assert set(required).issubset(df.columns)
    assert df["source_row_id"].is_unique
    assert df["partition"].isin(PARTITIONS).all()
    assert df[TARGET].isin([0, 1]).all()

baseline_assignments = (
    baseline[["source_row_id", "partition"]]
    .sort_values("source_row_id")
    .reset_index(drop=True)
)

expanded_assignments = (
    expanded[["source_row_id", "partition"]]
    .sort_values("source_row_id")
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    baseline_assignments,
    expanded_assignments
)

assert set(baseline["source_row_id"]) == set(
    expanded["source_row_id"]
)

# ============================================================
# 4. PREPARE THE m14_1 UNKNOWN INDICATOR
# ============================================================

expanded = expanded.copy()

expanded["m14_1"] = pd.to_numeric(
    expanded["m14_1"],
    errors="raise"
)

# Confirm only valid counts or the DHS don't-know code.
valid_m14 = (
    expanded["m14_1"].between(0, 20)
    | expanded["m14_1"].eq(98)
)

assert valid_m14.all(), (
    "Unexpected m14_1 values found. Inspect before proceeding."
)

# Separate indicator for don't-know responses.
expanded["m14_1_unknown"] = (
    expanded["m14_1"].eq(98).astype(int)
)

# 98 is not a count; replace it with missing.
expanded.loc[
    expanded["m14_1"].eq(98), "m14_1"
] = np.nan

assert expanded["m14_1_unknown"].sum() == 314

# ============================================================
# 5. NORMALIZE CATEGORICAL DATA TYPES
# ============================================================

for col in BASELINE_CATEGORICAL:
    baseline[col] = baseline[col].astype("string")

for col in EXPANDED_CATEGORICAL:
    expanded[col] = expanded[col].astype("string")

# The v701 categories must remain distinct.
expected_v701 = {
    "no education",
    "primary",
    "secondary",
    "higher",
    "don't know",
    "not applicable - no current union"
}

assert set(expanded["v701"].dropna().unique()) == expected_v701

# ============================================================
# 6. BUILD PREPROCESSING PIPELINE
# ============================================================

def make_one_hot_encoder():

    # Support both newer and older sklearn versions.
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            dtype=np.float32
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
            dtype=np.float32
        )


def build_preprocessor(
    numeric_features,
    categorical_features,
    include_m14_unknown=False
):

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder())
    ])

    transformers = [
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]

    if include_m14_unknown:
        transformers.append(
            ("m14_unknown", "passthrough", ["m14_1_unknown"])
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=True
    )


# ============================================================
# 7. FIT ON TRAIN ONLY; TRANSFORM VALIDATION AND TEST
# ============================================================

configurations = {
    "baseline": {
        "data": baseline,
        "numeric": BASELINE_NUMERIC,
        "categorical": BASELINE_CATEGORICAL,
        "include_m14_unknown": False
    },
    "expanded": {
        "data": expanded,
        "numeric": EXPANDED_NUMERIC,
        "categorical": EXPANDED_CATEGORICAL,
        "include_m14_unknown": True
    }
}

manifest = {
    "dataset": "2018 NDHS Individual Recode",
    "split_seed": SEED,
    "preprocessing_fit_partition": "train",
    "validation_used_for_fitting": False,
    "test_used_for_fitting": False,
    "target": TARGET,
    "configurations": {},
    "sklearn_version": sklearn.__version__
}

for config_name, config in configurations.items():

    print("\n" + "=" * 65)
    print("PREPROCESSING:", config_name.upper())
    print("=" * 65)

    df = config["data"]

    feature_columns = (
        config["numeric"]
        + config["categorical"]
    )

    if config["include_m14_unknown"]:
        feature_columns += ["m14_1_unknown"]

    # Partition views.
    train_df = df.loc[
        df["partition"] == "train"
    ].copy()

    validation_df = df.loc[
        df["partition"] == "validation"
    ].copy()

    test_df = df.loc[
        df["partition"] == "test"
    ].copy()

    assert len(train_df) > 0
    assert len(validation_df) > 0
    assert len(test_df) > 0

    # Build and fit the preprocessor using training data only.
    preprocessor = build_preprocessor(
        numeric_features=config["numeric"],
        categorical_features=config["categorical"],
        include_m14_unknown=config["include_m14_unknown"]
    )

    X_train = preprocessor.fit_transform(
        train_df[feature_columns]
    )

    # Apply the already-fitted transformation.
    X_validation = preprocessor.transform(
        validation_df[feature_columns]
    )

    X_test = preprocessor.transform(
        test_df[feature_columns]
    )

    # Convert to float32 for PyTorch compatibility.
    X_train = np.asarray(X_train, dtype=np.float32)
    X_validation = np.asarray(
        X_validation, dtype=np.float32
    )
    X_test = np.asarray(X_test, dtype=np.float32)

    y_train = train_df[TARGET].to_numpy(dtype=np.float32)
    y_validation = validation_df[TARGET].to_numpy(
        dtype=np.float32
    )
    y_test = test_df[TARGET].to_numpy(dtype=np.float32)

    # --------------------------------------------------------
    # Integrity checks
    # --------------------------------------------------------

    assert X_train.shape[1] == X_validation.shape[1]
    assert X_train.shape[1] == X_test.shape[1]

    assert np.isfinite(X_train).all()
    assert np.isfinite(X_validation).all()
    assert np.isfinite(X_test).all()

    assert set(np.unique(y_train)) == {0.0, 1.0}
    assert set(np.unique(y_validation)) == {0.0, 1.0}
    assert set(np.unique(y_test)) == {0.0, 1.0}

    feature_names = (
        preprocessor.get_feature_names_out().tolist()
    )

    assert len(feature_names) == X_train.shape[1]
    assert len(feature_names) == len(set(feature_names))

    # --------------------------------------------------------
    # Save fitted preprocessor
    # --------------------------------------------------------

    joblib.dump(
        preprocessor,
        OUTPUT_DIR / f"{config_name}_preprocessor.joblib"
    )

    # Save transformed arrays and respondent identifiers.
    for partition_name, partition_df, X, y in [
        ("train", train_df, X_train, y_train),
        ("validation", validation_df, X_validation, y_validation),
        ("test", test_df, X_test, y_test)
    ]:

        np.savez_compressed(
            OUTPUT_DIR / f"{config_name}_{partition_name}.npz",
            X=X,
            y=y,
            source_row_id=partition_df[
                "source_row_id"
            ].astype(str).to_numpy(),
            v024=partition_df["v024"].to_numpy()
        )

    # Save feature names.
    pd.DataFrame({
        "feature_index": range(len(feature_names)),
        "processed_feature": feature_names
    }).to_csv(
        OUTPUT_DIR / f"{config_name}_feature_names.csv",
        index=False
    )

    # Record configuration.
    manifest["configurations"][config_name] = {
        "input_features": feature_columns,
        "numeric_features": config["numeric"],
        "categorical_features": config["categorical"],
        "processed_input_dimension": int(X_train.shape[1]),
        "train_shape": list(X_train.shape),
        "validation_shape": list(X_validation.shape),
        "test_shape": list(X_test.shape),
        "train_class_counts": {
            "home": int((y_train == 0).sum()),
            "facility": int((y_train == 1).sum())
        },
        "validation_class_counts": {
            "home": int((y_validation == 0).sum()),
            "facility": int((y_validation == 1).sum())
        },
        "test_class_counts": {
            "home": int((y_test == 0).sum()),
            "facility": int((y_test == 1).sum())
        },
        "all_values_finite": True,
        "feature_names": feature_names
    }

    print("Training matrix:", X_train.shape)
    print("Validation matrix:", X_validation.shape)
    print("Test matrix:", X_test.shape)
    print("Processed input dimension:", X_train.shape[1])
    print("All values finite: PASS")
    print("Consistent dimensions: PASS")

# ============================================================
# 8. SAVE MANIFEST
# ============================================================

manifest["respondent_assignments_shared"] = True
manifest["model_training_performed"] = False
manifest["test_used_for_model_selection"] = False

with open(
    OUTPUT_DIR / "preprocessing_manifest.json",
    "w"
) as f:
    json.dump(manifest, f, indent=4)

print("\n" + "=" * 65)
print("STEP 5 PREPROCESSING COMPLETED")
print("=" * 65)
print("Output directory:", OUTPUT_DIR)
print("No model training has been performed.")



PREPROCESSING: BASELINE
Training matrix: (12851, 15)
Validation matrix: (4299, 15)
Test matrix: (4292, 15)
Processed input dimension: 15
All values finite: PASS
Consistent dimensions: PASS

PREPROCESSING: EXPANDED
Training matrix: (12851, 51)
Validation matrix: (4299, 51)
Test matrix: (4292, 51)
Processed input dimension: 51
All values finite: PASS
Consistent dimensions: PASS

STEP 5 PREPROCESSING COMPLETED
Output directory: C:\Users\usher\Documents\NDHS-Fl-Research\NDHS_FL_Research\05_preprocessed
No model training has been performed.


## client-level audit: baseline

In [10]:
PREPROCESS = ROOT / "05_preprocessed"

regions = {
    1: "North Central",
    2: "North East",
    3: "North West",
    4: "South East",
    5: "South South",
    6: "South West",
}

results = []

for config, expected_features in [
    ("baseline", 15),
    ("expanded", 51)
]:
    print("\n" + "=" * 65)
    print(f"{config.upper()} REGIONAL CLIENT AUDIT")
    print("=" * 65)

    for partition in ["train", "validation", "test"]:
        path = PREPROCESS / f"{config}_{partition}.npz"

        with np.load(path, allow_pickle=True) as data:
            X = data["X"]
            y = data["y"].astype(int)
            region_ids = data["v024"].astype(int)

        assert X.shape[1] == expected_features
        assert np.isfinite(X).all()
        assert set(np.unique(y)) == {0, 1}

        print(f"\n{partition.upper()} — matrix {X.shape}")

        for region_id, region_name in regions.items():
            mask = region_ids == region_id

            X_client = X[mask]
            y_client = y[mask]

            assert len(y_client) > 0
            assert set(np.unique(y_client)) == {0, 1}

            home = int((y_client == 0).sum())
            facility = int((y_client == 1).sum())

            print(
                f"{region_name:15s} "
                f"n={len(y_client):5d} "
                f"home={home:5d} "
                f"facility={facility:5d}"
            )

            results.append({
                "configuration": config,
                "partition": partition,
                "region": region_name,
                "n": len(y_client),
                "home": home,
                "facility": facility,
                "features": X_client.shape[1],
            })

print("\n" + "=" * 65)
print("STEP 6 FINAL AUDIT")
print("=" * 65)

audit = pd.DataFrame(results)

assert len(audit) == 36
assert (audit["home"] > 0).all()
assert (audit["facility"] > 0).all()

audit.to_csv(
    PREPROCESS / "client_integrity_audit.csv",
    index=False
)

print("36 regional client-partition checks: PASS")
print("Both outcome classes present everywhere: PASS")
print("Feature dimensions: PASS")
print("Audit saved successfully.")



BASELINE REGIONAL CLIENT AUDIT

TRAIN — matrix (12851, 15)
North Central   n= 2304 home= 1105 facility= 1199
North East      n= 2698 home= 1997 facility=  701
North West      n= 3767 home= 3174 facility=  593
South East      n= 1402 home=  271 facility= 1131
South South     n= 1231 home=  632 facility=  599
South West      n= 1449 home=  272 facility= 1177

VALIDATION — matrix (4299, 15)
North Central   n=  768 home=  369 facility=  399
North East      n=  903 home=  667 facility=  236
North West      n= 1269 home= 1069 facility=  200
South East      n=  470 home=   90 facility=  380
South South     n=  408 home=  209 facility=  199
South West      n=  481 home=   89 facility=  392

TEST — matrix (4292, 15)
North Central   n=  766 home=  367 facility=  399
North East      n=  897 home=  665 facility=  232
North West      n= 1271 home= 1075 facility=  196
South East      n=  466 home=   90 facility=  376
South South     n=  411 home=  211 facility=  200
South West      n=  481 home=   

## controlled runtime pilot
the purpose of this pilot is to estimate computatonal cost and verify that the training implementation works correctly. It is not a perfromance experiment, and its results is not reported as confirmatory finds

In [11]:
from torch.utils.data import DataLoader, TensorDataset
import time

# ============================================================
# 1. CONFIGURATION
# ============================================================

DATA_DIR = ROOT / "05_preprocessed"
OUTPUT_DIR = ROOT / "06_runtime_pilot"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
ROUNDS = 3
LOCAL_EPOCHS = 1
BATCH_SIZE = 32
LEARNING_RATE = 0.001
FEDPROX_MU = 0.01

REGIONS = {
    1: "North Central",
    2: "North East",
    3: "North West",
    4: "South East",
    5: "South South",
    6: "South West",
}

DEVICE = torch.device("cpu")

# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(True)

set_seed(SEED)

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

# ============================================================
# 3. LOAD DATA
# ============================================================

def load_partition(config, partition):
    path = DATA_DIR / f"{config}_{partition}.npz"

    with np.load(path, allow_pickle=True) as data:
        X = np.asarray(data["X"], dtype=np.float32)
        y = np.asarray(data["y"], dtype=np.float32).reshape(-1)
        regions = np.asarray(data["v024"], dtype=np.int64)

    assert np.isfinite(X).all()
    assert len(X) == len(y) == len(regions)

    return X, y, regions

# ============================================================
# 4. MODEL
# ============================================================

class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(1)

# ============================================================
# 5. LOCAL CLIENT TRAINING
# ============================================================

def train_client(
    global_model,
    X_client,
    y_client,
    mu,
    client_seed
):
    set_seed(client_seed)

    X_tensor = torch.tensor(X_client, dtype=torch.float32)
    y_tensor = torch.tensor(y_client, dtype=torch.float32)

    generator = torch.Generator()
    generator.manual_seed(client_seed)

    loader = DataLoader(
        TensorDataset(X_tensor, y_tensor),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0
    )

    local_model = MLP(X_client.shape[1]).to(DEVICE)
    local_model.load_state_dict(global_model.state_dict())

    # Reference parameters remain fixed during local training.
    global_parameters = [
        parameter.detach().clone()
        for parameter in global_model.parameters()
    ]

    optimizer = torch.optim.Adam(
        local_model.parameters(),
        lr=LEARNING_RATE
    )

    criterion = nn.BCEWithLogitsLoss()

    local_model.train()

    for epoch in range(LOCAL_EPOCHS):
        for batch_X, batch_y in loader:
            batch_X = batch_X.to(DEVICE)
            batch_y = batch_y.to(DEVICE)

            optimizer.zero_grad()

            logits = local_model(batch_X)
            loss = criterion(logits, batch_y)

            if mu > 0:
                proximal_penalty = torch.zeros(
                    (), device=DEVICE
                )

                for local_parameter, global_parameter in zip(
                    local_model.parameters(),
                    global_parameters
                ):
                    proximal_penalty += torch.sum(
                        (local_parameter - global_parameter) ** 2
                    )

                loss = loss + (mu / 2.0) * proximal_penalty

            loss.backward()
            optimizer.step()

    return local_model.state_dict()

# ============================================================
# 6. FEDERATED AGGREGATION
# ============================================================

def aggregate_models(client_states, client_sizes):
    total_samples = sum(client_sizes)

    global_state = {}

    for key in client_states[0]:
        global_state[key] = torch.zeros_like(
            client_states[0][key]
        )

        for state, size in zip(client_states, client_sizes):
            global_state[key] += (
                state[key] * (size / total_samples)
            )

    return global_state

# ============================================================
# 7. VALIDATION EVALUATION
# ============================================================

def evaluate_model(model, X, y, regions):
    model.eval()

    with torch.no_grad():
        X_tensor = torch.tensor(X, dtype=torch.float32)
        y_tensor = torch.tensor(y, dtype=torch.float32)

        logits = model(X_tensor)
        probabilities = torch.sigmoid(logits)

        criterion = nn.BCEWithLogitsLoss()
        loss = criterion(logits, y_tensor).item()

        predictions = (probabilities >= 0.5).int().numpy()
        actual = y.astype(int)

    accuracy = float((predictions == actual).mean())

    results = [{
        "region_id": 0,
        "region": "Pooled",
        "n": len(actual),
        "validation_loss": loss,
        "accuracy": accuracy
    }]

    for region_id, region_name in REGIONS.items():
        mask = regions == region_id

        regional_accuracy = float(
            (predictions[mask] == actual[mask]).mean()
        )

        results.append({
            "region_id": region_id,
            "region": region_name,
            "n": int(mask.sum()),
            "validation_loss": float(
                nn.functional.binary_cross_entropy_with_logits(
                    logits[mask],
                    y_tensor[mask]
                ).item()
            ),
            "accuracy": regional_accuracy
        })

    return results

# ============================================================
# 8. PILOT EXPERIMENT
# ============================================================

all_round_results = []
all_validation_results = []
runtime_results = []

for config in ["baseline", "expanded"]:

    X_train, y_train, train_regions = load_partition(
        config, "train"
    )

    X_val, y_val, val_regions = load_partition(
        config, "validation"
    )

    input_dim = X_train.shape[1]

    print("\n" + "=" * 65)
    print("CONFIGURATION:", config.upper())
    print("Input dimension:", input_dim)
    print("=" * 65)

    for algorithm in ["FedAvg", "FedProx"]:

        mu = 0.0 if algorithm == "FedAvg" else FEDPROX_MU

        # Reset to the same initialization for both algorithms
        set_seed(SEED)

        global_model = MLP(input_dim).to(DEVICE)

        initial_parameters = sum(
            parameter.numel()
            for parameter in global_model.parameters()
        )

        print(
            f"\nStarting {algorithm} | "
            f"parameters={initial_parameters} | mu={mu}"
        )

        experiment_start = time.perf_counter()

        for round_number in range(1, ROUNDS + 1):

            round_start = time.perf_counter()

            client_states = []
            client_sizes = []

            for region_id in REGIONS:

                mask = train_regions == region_id

                X_client = X_train[mask]
                y_client = y_train[mask]

                assert len(y_client) > 0

                client_seed = (
                    SEED
                    + round_number * 1000
                    + region_id
                )

                client_state = train_client(
                    global_model,
                    X_client,
                    y_client,
                    mu,
                    client_seed
                )

                client_states.append(client_state)
                client_sizes.append(len(y_client))

            # Sample-size weighted aggregation
            new_global_state = aggregate_models(
                client_states,
                client_sizes
            )

            global_model.load_state_dict(new_global_state)

            round_seconds = (
                time.perf_counter() - round_start
            )

            # Validation only; test partition is not loaded.
            validation_metrics = evaluate_model(
                global_model,
                X_val,
                y_val,
                val_regions
            )

            for metric in validation_metrics:
                all_validation_results.append({
                    "configuration": config,
                    "algorithm": algorithm,
                    "seed": SEED,
                    "round": round_number,
                    **metric
                })

            pooled = validation_metrics[0]

            all_round_results.append({
                "configuration": config,
                "algorithm": algorithm,
                "seed": SEED,
                "round": round_number,
                "round_seconds": round_seconds,
                "validation_loss": pooled["validation_loss"],
                "validation_accuracy": pooled["accuracy"],
                "input_dimension": input_dim,
                "parameters": initial_parameters
            })

            print(
                f"Round {round_number}/{ROUNDS} | "
                f"time={round_seconds:.2f}s | "
                f"val_loss={pooled['validation_loss']:.4f} | "
                f"val_accuracy={pooled['accuracy']:.4f}"
            )

        total_seconds = (
            time.perf_counter() - experiment_start
        )

        runtime_results.append({
            "configuration": config,
            "algorithm": algorithm,
            "seed": SEED,
            "rounds": ROUNDS,
            "local_epochs": LOCAL_EPOCHS,
            "total_seconds": total_seconds,
            "mean_round_seconds": (
                total_seconds / ROUNDS
            ),
            "input_dimension": input_dim,
            "parameters": initial_parameters
        })

        print(
            f"{algorithm} completed in "
            f"{total_seconds:.2f} seconds."
        )

# ============================================================
# 9. SAVE PILOT OUTPUTS
# ============================================================

round_df = pd.DataFrame(all_round_results)
validation_df = pd.DataFrame(all_validation_results)
runtime_df = pd.DataFrame(runtime_results)

round_df.to_csv(
    OUTPUT_DIR / "pilot_round_metrics.csv",
    index=False
)

validation_df.to_csv(
    OUTPUT_DIR / "pilot_regional_validation_metrics.csv",
    index=False
)

runtime_df.to_csv(
    OUTPUT_DIR / "pilot_runtime_summary.csv",
    index=False
)

pilot_manifest = {
    "purpose": "Runtime and implementation pilot only",
    "seed": SEED,
    "rounds": ROUNDS,
    "local_epochs": LOCAL_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "fedprox_mu": FEDPROX_MU,
    "device": str(DEVICE),
    "algorithms": ["FedAvg", "FedProx"],
    "feature_configurations": ["baseline", "expanded"],
    "evaluation_partition": "validation",
    "test_partition_accessed": False,
    "confirmatory_experiment": False
}

with open(
    OUTPUT_DIR / "pilot_manifest.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(pilot_manifest, f, indent=4)

print("\n" + "=" * 65)
print("RUNTIME PILOT COMPLETED")
print("=" * 65)

print(runtime_df.to_string(index=False))
print("\nOutputs saved to:", OUTPUT_DIR)


Device: cpu
PyTorch: 2.14.0+cpu

CONFIGURATION: BASELINE
Input dimension: 15

Starting FedAvg | parameters=401 | mu=0.0
Round 1/3 | time=1.27s | val_loss=0.6599 | val_accuracy=0.5808
Round 2/3 | time=1.22s | val_loss=0.6056 | val_accuracy=0.6439
Round 3/3 | time=1.34s | val_loss=0.5561 | val_accuracy=0.7406
FedAvg completed in 3.85 seconds.

Starting FedProx | parameters=401 | mu=0.01
Round 1/3 | time=1.62s | val_loss=0.6611 | val_accuracy=0.5801
Round 2/3 | time=2.07s | val_loss=0.6083 | val_accuracy=0.6239
Round 3/3 | time=2.08s | val_loss=0.5591 | val_accuracy=0.7358
FedProx completed in 5.79 seconds.

CONFIGURATION: EXPANDED
Input dimension: 51

Starting FedAvg | parameters=977 | mu=0.0
Round 1/3 | time=1.37s | val_loss=0.6416 | val_accuracy=0.5799
Round 2/3 | time=1.30s | val_loss=0.5692 | val_accuracy=0.6816
Round 3/3 | time=1.30s | val_loss=0.4961 | val_accuracy=0.7853
FedAvg completed in 3.99 seconds.

Starting FedProx | parameters=977 | mu=0.01
Round 1/3 | time=1.85s | val_los

In [12]:
from pathlib import Path
import pandas as pd

ROOT = Path(
    r"C:\Users\usher\Documents\NDHS-Fl-Research\NDHS_FL_Research"
)

PILOT_DIR = ROOT / "06_runtime_pilot"

df = pd.read_csv(
    PILOT_DIR / "pilot_regional_validation_metrics.csv"
)

# Regional majority-class accuracy from the fixed validation split.
majority_baselines = {
    "North Central": 399 / 768,
    "North East": 667 / 903,
    "North West": 1069 / 1269,
    "South East": 380 / 470,
    "South South": 209 / 408,
    "South West": 392 / 481,
}

df["majority_class_accuracy"] = df["region"].map(
    majority_baselines
)

df["accuracy_minus_majority"] = (
    df["accuracy"] - df["majority_class_accuracy"]
)

print("\nROUND 3 REGIONAL VALIDATION RESULTS")
print("=" * 90)

round3 = df[df["round"] == 3].copy()

print(
    round3[
        [
            "configuration",
            "algorithm",
            "region",
            "n",
            "accuracy",
            "majority_class_accuracy",
            "accuracy_minus_majority",
            "validation_loss"
        ]
    ].to_string(index=False)
)

print("\nPOOLED VALIDATION TRAJECTORIES")
print("=" * 90)

pooled = df[df["region"] == "Pooled"]

print(
    pooled[
        [
            "configuration",
            "algorithm",
            "round",
            "validation_loss",
            "accuracy"
        ]
    ].to_string(index=False)
)


ROUND 3 REGIONAL VALIDATION RESULTS
configuration algorithm        region    n  accuracy  majority_class_accuracy  accuracy_minus_majority  validation_loss
     baseline    FedAvg        Pooled 4299  0.740637                      NaN                      NaN         0.556101
     baseline    FedAvg North Central  768  0.613281                 0.519531                 0.093750         0.648421
     baseline    FedAvg    North East  903  0.768549                 0.738649                 0.029900         0.509927
     baseline    FedAvg    North West 1269  0.858156                 0.842396                 0.015760         0.426976
     baseline    FedAvg    South East  470  0.682979                 0.808511                -0.125532         0.663007
     baseline    FedAvg   South South  408  0.691176                 0.512255                 0.178922         0.638047
     baseline    FedAvg    South West  481  0.679834                 0.814969                -0.135135         0.662074
   

## Baseline 15-feature MMD2

In [13]:
# ============================================================
# PHASE 3 — BASELINE FEATURE HETEROGENEITY (15 FEATURES)
# Fresh, reproducible MMD² analysis
# ============================================================

DATA_DIR = ROOT / "05_preprocessed"
OUTPUT_DIR = ROOT / "07_heterogeneity"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = DATA_DIR / "baseline_train.npz"

# Prespecified analysis settings
EXPECTED_FEATURES = 15
SAMPLE_SIZE = 1000
RBF_GAMMA = 0.02
BASE_SEED = 2026

REGIONS = {
    1: "North Central",
    2: "North East",
    3: "North West",
    4: "South East",
    5: "South South",
    6: "South West"
}

# ============================================================
# 1. LOAD TRAINING DATA ONLY
# ============================================================

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Training matrix not found: {INPUT_FILE}")

with np.load(INPUT_FILE, allow_pickle=True) as data:
    X = np.asarray(data["X"], dtype=np.float64)
    regions = np.asarray(data["v024"], dtype=np.int64)

assert X.ndim == 2, "X must be a two-dimensional matrix"
assert X.shape[1] == EXPECTED_FEATURES, (
    f"Expected {EXPECTED_FEATURES} featrures, found {X.shape[1]}"
)
assert len(X) == len(regions), "X and region arrays have different lengths"
assert np.isfinite(X).all(), "Training matrix contains non-finite values"
assert set(np.unique(regions)) == set(REGIONS), (
    "Trainign data does not contain exactly the six expected regions"
)

print("BASELINR TRAINING DATA")
print("=" * 50)
print("Rows:", len(X))
print("Processed features:", X.shape[1])
print("Regions:", len(np.unique(regions)))

# ============================================================
# 2. RBF KERNEL
# ============================================================

def rbf_kernel_matrix(A, B, gamma):
    """
    RBF kernel:
    K(x, y) = exp(-gamma * ||x-y||²)
    """
    squared_distances = cdist(A, B, metric="sqeuclidean")
    return np.exp(-gamma * squared_distances)


# ============================================================
# 3. BIASED EMPIRICAL MMD² ESTIMATOR
# ============================================================

def calculate_mmd2(X_client, X_reference, gamma):
    """
    Biased empirical MMD² (V-statistic):

      MMD² = mean(Kxx) + mean(Kyy) - 2*mean(Kxy)

    Includes diagonal terms in Kxx and Kyy.
    """

    K_xx = rbf_kernel_matrix(X_client, X_client, gamma)
    K_yy = rbf_kernel_matrix(X_reference, X_reference, gamma)
    K_xy = rbf_kernel_matrix(X_client, X_reference, gamma)

    mmd2 = (
        K_xx.mean()
        + K_yy.mean()
        - 2.0 * K_xy.mean()
    )

    return float(mmd2)


# ============================================================
# 4. ZONE-VERSUS-OTHER-FIVE-ZONES COMPARISON
# ============================================================

results = []

for region_id, region_name in REGIONS.items():

    client_indices = np.flatnonzero(regions == region_id)
    reference_indices = np.flatnonzero(regions != region_id)

    n_client_available = len(client_indices)
    n_reference_available = len(reference_indices)

    if n_client_available < SAMPLE_SIZE:
        raise ValueError(
            f"{region_name} has only {n_client_available} "
            f"training observations."
        )

    if n_reference_available < SAMPLE_SIZE:
        raise ValueError(
            f"Reference group for {region_name} has only "
            f"{n_reference_available} observations."
        )

    # Independent, reproducible sampling for each zone.
    rng = np.random.default_rng(BASE_SEED + region_id)

    sampled_client_indices = rng.choice(
        client_indices,
        size=SAMPLE_SIZE,
        replace=False
    )

    sampled_reference_indices = rng.choice(
        reference_indices,
        size=SAMPLE_SIZE,
        replace=False
    )

    X_client = X[sampled_client_indices]
    X_reference = X[sampled_reference_indices]

    mmd2 = calculate_mmd2(
        X_client,
        X_reference,
        gamma=RBF_GAMMA
    )

    results.append({
        "region_id": region_id,
        "region": region_name,
        "mmd2": mmd2,
        "client_sample_size": SAMPLE_SIZE,
        "reference_sample_size": SAMPLE_SIZE,
        "client_training_n": n_client_available,
        "other_five_zones_training_n": n_reference_available,
        "feature_dimension": X.shape[1],
        "rbf_gamma": RBF_GAMMA,
        "sampling_seed": BASE_SEED + region_id,
        "estimator": "biased_empirical_v_statistic",
    })

    print(
        f"{region_name:18s} | "
        f"MMD² = {mmd2:.8f} | "
        f"client n = {n_client_available} | "
        f"reference n = {n_reference_available}"
    )

# ============================================================
# 5. SAVE RESULTS
# ============================================================

results_df = pd.DataFrame(results)

results_file = OUTPUT_DIR / "baseline_15feature_mmd2.csv"

results_df.to_csv(results_file, index=False)

manifest = {
    "analysis_id": "PHASE3_BASELINE_MMD_001",
    "purpose": "Baseline feature-distribution heterogeneity",
    "input_file": str(INPUT_FILE),
    "data_partition": "training_only",
    "feature_configuration": "baseline",
    "feature_dimension": int(X.shape[1]),
    "sample_size_per_group": SAMPLE_SIZE,
    "sampling": "without_replacement",
    "sampling_seed_base": BASE_SEED,
    "sampling_seed_by_region": {
        REGIONS[rid]: BASE_SEED + rid
        for rid in REGIONS
    },
    "reference_definition": (
        "Each region compared against the pooled other five regions"
    ),
    "kernel": "RBF",
    "rbf_gamma": RBF_GAMMA,
    "mmd_estimator": "biased_empirical_v_statistic",
    "estimator_formula": (
        "mean(Kxx) + mean(Kyy) - 2*mean(Kxy), "
        "including diagonal terms"
    ),
    "regions": list(REGIONS.values()),
    "test_partition_accessed": False,
    "validation_partition_accessed": False,
    "historical_mmd_results_reproduced": False,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}

manifest_file = OUTPUT_DIR / "baseline_15feature_mmd2_manifest.json"

with open(manifest_file, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

print("\n" + "=" * 65)
print("BASELINE MMD² ANALYSIS COMPLETED")
print("=" * 65)

print(results_df[
    ["region", "mmd2", "feature_dimension", "rbf_gamma"]
].to_string(index=False))

print("\nResults saved to:", results_file)
print("Manifest saved to:", manifest_file)


BASELINR TRAINING DATA
Rows: 12851
Processed features: 15
Regions: 6
North Central      | MMD² = 0.00232723 | client n = 2304 | reference n = 10547
North East         | MMD² = 0.01028590 | client n = 2698 | reference n = 10153
North West         | MMD² = 0.01725742 | client n = 3767 | reference n = 9084
South East         | MMD² = 0.02946542 | client n = 1402 | reference n = 11449
South South        | MMD² = 0.01697773 | client n = 1231 | reference n = 11620
South West         | MMD² = 0.03435187 | client n = 1449 | reference n = 11402

BASELINE MMD² ANALYSIS COMPLETED
       region     mmd2  feature_dimension  rbf_gamma
North Central 0.002327                 15       0.02
   North East 0.010286                 15       0.02
   North West 0.017257                 15       0.02
   South East 0.029465                 15       0.02
  South South 0.016978                 15       0.02
   South West 0.034352                 15       0.02

Results saved to: C:\Users\usher\Documents\NDHS-Fl-R

## Expanded 51-features MMD2

In [14]:
# ============================================================
# PHASE 3 — EXPANDED FEATURE HETEROGENEITY (51 FEATURES)
# Fresh, reproducible MMD² analysis
# ============================================================

ROOT = Path(
    r"C:\Users\usher\Documents\NDHS-Fl-Research\NDHS_FL_Research"
)

DATA_DIR = ROOT / "05_preprocessed"
OUTPUT_DIR = ROOT / "07_heterogeneity"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = DATA_DIR / "expanded_train.npz"

# Prespecified analysis settings
EXPECTED_FEATURES = 51
SAMPLE_SIZE = 1000
RBF_GAMMA = 0.02
BASE_SEED = 2026

REGIONS = {
    1: "North Central",
    2: "North East",
    3: "North West",
    4: "South East",
    5: "South South",
    6: "South West",
}

# ============================================================
# 1. LOAD TRAINING DATA ONLY
# ============================================================

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Training matrix not found: {INPUT_FILE}"
    )

with np.load(INPUT_FILE, allow_pickle=True) as data:
    X = np.asarray(data["X"], dtype=np.float64)
    regions = np.asarray(data["v024"], dtype=np.int64)

assert X.ndim == 2, "X must be a two-dimensional matrix."

assert X.shape[1] == EXPECTED_FEATURES, (
    f"Expected {EXPECTED_FEATURES} features, "
    f"found {X.shape[1]}."
)

assert len(X) == len(regions), (
    "X and region arrays have different lengths."
)

assert np.isfinite(X).all(), (
    "Training matrix contains non-finite values."
)

assert set(np.unique(regions)) == set(REGIONS), (
    "Training data does not contain exactly "
    "the six expected regions."
)

print("EXPANDED TRAINING DATA")
print("=" * 65)
print("Rows:", len(X))
print("Processed features:", X.shape[1])
print("Regions:", len(np.unique(regions)))

# ============================================================
# 2. RBF KERNEL
# ============================================================

def rbf_kernel_matrix(A, B, gamma):
    """
    RBF kernel:
        K(x, y) = exp(-gamma * ||x-y||²)
    """
    squared_distances = cdist(
        A,
        B,
        metric="sqeuclidean"
    )

    return np.exp(-gamma * squared_distances)


# ============================================================
# 3. BIASED EMPIRICAL MMD² ESTIMATOR
# ============================================================

def calculate_mmd2(X_client, X_reference, gamma):
    """
    Biased empirical MMD² (V-statistic):

      MMD² = mean(Kxx) + mean(Kyy) - 2*mean(Kxy)

    Includes diagonal terms in Kxx and Kyy.
    """

    K_xx = rbf_kernel_matrix(
        X_client,
        X_client,
        gamma
    )

    K_yy = rbf_kernel_matrix(
        X_reference,
        X_reference,
        gamma
    )

    K_xy = rbf_kernel_matrix(
        X_client,
        X_reference,
        gamma
    )

    mmd2 = (
        K_xx.mean()
        + K_yy.mean()
        - 2.0 * K_xy.mean()
    )

    return float(mmd2)


# ============================================================
# 4. ZONE-VERSUS-OTHER-FIVE-ZONES COMPARISON
# ============================================================

results = []

for region_id, region_name in REGIONS.items():

    client_indices = np.flatnonzero(
        regions == region_id
    )

    reference_indices = np.flatnonzero(
        regions != region_id
    )

    n_client_available = len(client_indices)
    n_reference_available = len(reference_indices)

    if n_client_available < SAMPLE_SIZE:
        raise ValueError(
            f"{region_name} has only "
            f"{n_client_available} training observations."
        )

    if n_reference_available < SAMPLE_SIZE:
        raise ValueError(
            f"Reference group for {region_name} has only "
            f"{n_reference_available} observations."
        )

    # Independent, reproducible sampling for each zone.
    rng = np.random.default_rng(
        BASE_SEED + region_id
    )

    sampled_client_indices = rng.choice(
        client_indices,
        size=SAMPLE_SIZE,
        replace=False
    )

    sampled_reference_indices = rng.choice(
        reference_indices,
        size=SAMPLE_SIZE,
        replace=False
    )

    X_client = X[sampled_client_indices]
    X_reference = X[sampled_reference_indices]

    mmd2 = calculate_mmd2(
        X_client,
        X_reference,
        gamma=RBF_GAMMA
    )

    results.append({
        "region_id": region_id,
        "region": region_name,
        "mmd2": mmd2,
        "client_sample_size": SAMPLE_SIZE,
        "reference_sample_size": SAMPLE_SIZE,
        "client_training_n": n_client_available,
        "other_five_zones_training_n": n_reference_available,
        "feature_dimension": X.shape[1],
        "rbf_gamma": RBF_GAMMA,
        "sampling_seed": BASE_SEED + region_id,
        "estimator": "biased_empirical_v_statistic",
    })

    print(
        f"{region_name:18s} | "
        f"MMD² = {mmd2:.8f} | "
        f"client n = {n_client_available} | "
        f"reference n = {n_reference_available}"
    )

# ============================================================
# 5. SAVE RESULTS
# ============================================================

results_df = pd.DataFrame(results)

results_file = (
    OUTPUT_DIR / "expanded_51feature_mmd2.csv"
)

results_df.to_csv(
    results_file,
    index=False
)

manifest = {
    "analysis_id": "PHASE3_EXPANDED_MMD_001",
    "purpose": "Expanded feature-distribution heterogeneity",
    "input_file": str(INPUT_FILE),
    "data_partition": "training_only",
    "feature_configuration": "expanded",
    "feature_dimension": int(X.shape[1]),
    "sample_size_per_group": SAMPLE_SIZE,
    "sampling": "without_replacement",
    "sampling_seed_base": BASE_SEED,
    "sampling_seed_by_region": {
        REGIONS[rid]: BASE_SEED + rid
        for rid in REGIONS
    },
    "reference_definition": (
        "Each region compared against the pooled other five regions"
    ),
    "kernel": "RBF",
    "rbf_gamma": RBF_GAMMA,
    "mmd_estimator": "biased_empirical_v_statistic",
    "estimator_formula": (
        "mean(Kxx) + mean(Kyy) - 2*mean(Kxy), "
        "including diagonal terms"
    ),
    "regions": list(REGIONS.values()),
    "test_partition_accessed": False,
    "validation_partition_accessed": False,
    "historical_mmd_results_reproduced": False,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

manifest_file = (
    OUTPUT_DIR / "expanded_51feature_mmd2_manifest.json"
)

with open(
    manifest_file,
    "w",
    encoding="utf-8"
) as f:
    json.dump(manifest, f, indent=4)

# ============================================================
# 6. DISPLAY FINAL RESULTS
# ============================================================

print("\n" + "=" * 65)
print("EXPANDED 51-FEATURE MMD² ANALYSIS COMPLETED")
print("=" * 65)

print(
    results_df[
        [
            "region",
            "mmd2",
            "feature_dimension",
            "rbf_gamma"
        ]
    ].to_string(index=False)
)

print("\nResults saved to:", results_file)
print("Manifest saved to:", manifest_file)


EXPANDED TRAINING DATA
Rows: 12851
Processed features: 51
Regions: 6
North Central      | MMD² = 0.01183577 | client n = 2304 | reference n = 10547
North East         | MMD² = 0.02952337 | client n = 2698 | reference n = 10153
North West         | MMD² = 0.06655617 | client n = 3767 | reference n = 9084
South East         | MMD² = 0.10269018 | client n = 1402 | reference n = 11449
South South        | MMD² = 0.05155573 | client n = 1231 | reference n = 11620
South West         | MMD² = 0.10323611 | client n = 1449 | reference n = 11402

EXPANDED 51-FEATURE MMD² ANALYSIS COMPLETED
       region     mmd2  feature_dimension  rbf_gamma
North Central 0.011836                 51       0.02
   North East 0.029523                 51       0.02
   North West 0.066556                 51       0.02
   South East 0.102690                 51       0.02
  South South 0.051556                 51       0.02
   South West 0.103236                 51       0.02

Results saved to: C:\Users\usher\Document

**finding 1 - Geographic feature heterogeneity has a consistent ordering**

Both new feature configurations produce exactly the same ordering:

1. South West
2. South East
3. North West
4. South South
5. North East
6. North Central

This is a useful consistency finding: the broad geographic patttern is not dependent on whether we use the baseline or expanded representation

**Finding 2 - Outcome and Feature heterogeneity are not identicical**

South West and South East rank highest in both outcome JS divergence and feature MMD².

However, South South has the lowest outcome JS divergence but a substantially larger feature MMD² than North Central and North East

This suggest that similar outcome prevalence does not necessarilty imply similar predictor distributions

**Finding 3 - Expanded features produces larger MMD² values**

The expanded reprensentation has higher MMD² values in every zone

That does not conclude that the geographic heterogeneity itself increased. The representation has changes from 15 to 51 dimensions;, which changes the geometry of the  RBF kernel and can change MMD² magnitudes. The consistent ranking is more defensible than comparing absolute values across the two representations.

**Finding 4 - These results do not establish result superiority**

I was able to identify zones with greater discrepancies, but I cannot yet say whether FedAvg or FedProx will handle thhose zones better. That reuires the later federated experiments


## correct and validate the federated Implementation

**Implementation conventions**

## 1. Model and training specifications

| Component | Canonical specification |
|---|---|
| Model | MLP |
| Architecture | Input → 16 → 8 → 1 |
| Hidden activation | ReLU |
| Output | Single logit |
| Loss | BCEWithLogitsLoss |
| Optimizer | Adam |
| Learning rate | 0.001 |
| Batch size | 32 |
| Optimizer lifecycle | Reset for each client-round |
| FedProx μ | 0.01 |
| Prediction threshold | 0.5 |

## 2. Federated learning specifications

| Component | Canonical specification |
|---|---|
| Clients | Six geopolitical zones |
| Participation | All six clients every round |
| Client ordering | Fixed region ID order, 1–6 |
| Aggregation | Sample-size-weighted FedAvg |
| FedProx reference | Fixed global parameters at start of local training |
| Client data | Existing frozen training partitions |
| Validation | Validation partition only |
| Test | Remains untouched until confirmatory evaluation |

## 3. Reproducibility specifications

- Seed Python, NumPy and PyTorch.
- Use explicit DataLoader generators.
- Keep `num_workers=0`.
- Derive client-round seeds from the experiment seed, round and region ID.
- Enable deterministic PyTorch algorithms.
- Preserve the saved training-matrix row order.
- Record software versions, device, configuration and seeds.

## 4. Sanity-test acceptance criteria

| Test | Proposed acceptance criterion |
|---|---|
| DET-01 | FedProx μ = 0 and FedAvg produce equivalent updates under matched conditions; maximum parameter difference ≤ $10^{-7}$. |
| DET-02 | Repeated same-seed runs have maximum parameter difference ≤ $10^{-7}$, with matching predictions and metrics within documented tolerance. |
| DET-03 | Initial model parameters match exactly. |

For the CPU environment, we will also record whether the results are bit-for-bit identical. The $10^{-7}$ tolerance is the acceptance threshold, not permission to overlook unexpected differences.

**Building the canonical implementation**

In [15]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# ------------------------------------------------------------
# 1. Existing region definitions
# ------------------------------------------------------------

REGIONS = {
    1: "North Central",
    2: "North East",
    3: "North West",
    4: "South East",
    5: "South South",
    6: "South West",
}

# ------------------------------------------------------------
# 2. Existing evaluation function, reproduced unchanged
# ------------------------------------------------------------

def evaluate_model(model, X, y, regions):
    model.eval()

    with torch.no_grad():
        X_tensor = torch.tensor(X, dtype=torch.float32)
        y_tensor = torch.tensor(y, dtype=torch.float32)

        logits = model(X_tensor)
        probabilities = torch.sigmoid(logits)

        criterion = nn.BCEWithLogitsLoss()
        loss = criterion(logits, y_tensor).item()

        predictions = (probabilities >= 0.5).int().numpy()
        actual = y.astype(int)

    accuracy = float((predictions == actual).mean())

    results = [{
        "region_id": 0,
        "region": "Pooled",
        "n": len(actual),
        "validation_loss": loss,
        "accuracy": accuracy
    }]

    for region_id, region_name in REGIONS.items():
        mask = regions == region_id

        regional_accuracy = float(
            (predictions[mask] == actual[mask]).mean()
        )

        results.append({
            "region_id": region_id,
            "region": region_name,
            "n": int(mask.sum()),
            "validation_loss": float(
                nn.functional.binary_cross_entropy_with_logits(
                    logits[mask],
                    y_tensor[mask]
                ).item()
            ),
            "accuracy": regional_accuracy
        })

    return results


# ------------------------------------------------------------
# 3. Controlled model: returns predetermined logits
# ------------------------------------------------------------

class ControlledLogitModel(nn.Module):
    def __init__(self, logits):
        super().__init__()
        self.register_buffer(
            "controlled_logits",
            torch.tensor(logits, dtype=torch.float32)
        )

    def forward(self, X):
        return self.controlled_logits


# Two observations per region.
# Each region has one negative and one positive case.
# Logits correspond to probabilities:
# 0.1, 0.8, 0.2, 0.9, 0.4, 0.7,
# repeated across the six regions.

probability_pattern = np.array(
    [0.1, 0.8, 0.2, 0.9, 0.4, 0.7],
    dtype=float
)

logit_pattern = np.log(
    probability_pattern / (1 - probability_pattern)
)

y_pattern = np.array([0, 0, 0, 1, 1, 1], dtype=int)

X = np.zeros((36, 1), dtype=np.float32)
y = np.tile(y_pattern, 6).astype(np.float32)

regions = np.repeat(
    np.arange(1, 7),
    6
)

controlled_logits = np.tile(logit_pattern, 6)

model = ControlledLogitModel(controlled_logits)

# ------------------------------------------------------------
# 4. Run the actual evaluation function
# ------------------------------------------------------------

results = evaluate_model(
    model,
    X,
    y,
    regions
)

results_df = pd.DataFrame(results)

print("CONTROLLED EVALUATION OUTPUT")
print("=" * 75)
print(results_df.to_string(index=False))

# ------------------------------------------------------------
# 5. Independent expected values
# ------------------------------------------------------------

expected_accuracy = 2 / 3

expected_pooled_loss = nn.functional.binary_cross_entropy_with_logits(
    torch.tensor(controlled_logits, dtype=torch.float32),
    torch.tensor(y, dtype=torch.float32)
).item()

checks = []

# Pooled checks
pooled = results[0]

checks.append({
    "check": "Pooled sample count",
    "expected": 36,
    "observed": pooled["n"],
    "passed": pooled["n"] == 36
})

checks.append({
    "check": "Pooled accuracy",
    "expected": expected_accuracy,
    "observed": pooled["accuracy"],
    "passed": np.isclose(
        pooled["accuracy"], expected_accuracy, atol=1e-7
    )
})

checks.append({
    "check": "Pooled validation loss",
    "expected": expected_pooled_loss,
    "observed": pooled["validation_loss"],
    "passed": np.isclose(
        pooled["validation_loss"],
        expected_pooled_loss,
        atol=1e-7
    )
})

# Regional checks
for result in results[1:]:
    checks.append({
        "check": f'{result["region"]}: sample count',
        "expected": 6,
        "observed": result["n"],
        "passed": result["n"] == 6
    })

    checks.append({
        "check": f'{result["region"]}: accuracy',
        "expected": expected_accuracy,
        "observed": result["accuracy"],
        "passed": np.isclose(
            result["accuracy"],
            expected_accuracy,
            atol=1e-7
        )
    })

checks_df = pd.DataFrame(checks)

print("\nCONTROLLED FUNCTION TESTS")
print("=" * 75)
print(checks_df.to_string(index=False))

assert checks_df["passed"].all(), (
    "At least one controlled evaluation test failed."
)

assert len(results) == 7
assert all(result["n"] > 0 for result in results)

print("\nPHASE 4E.2 NOTEBOOK FUNCTION TEST PASSED")


CONTROLLED EVALUATION OUTPUT
 region_id        region  n  validation_loss  accuracy
         0        Pooled 36         0.552711  0.666667
         1 North Central  6         0.552711  0.666667
         2    North East  6         0.552711  0.666667
         3    North West  6         0.552711  0.666667
         4    South East  6         0.552711  0.666667
         5   South South  6         0.552711  0.666667
         6    South West  6         0.552711  0.666667

CONTROLLED FUNCTION TESTS
                      check  expected  observed  passed
        Pooled sample count 36.000000 36.000000    True
            Pooled accuracy  0.666667  0.666667    True
     Pooled validation loss  0.552711  0.552711    True
North Central: sample count  6.000000  6.000000    True
    North Central: accuracy  0.666667  0.666667    True
   North East: sample count  6.000000  6.000000    True
       North East: accuracy  0.666667  0.666667    True
   North West: sample count  6.000000  6.000000    True


**Testing the actual function's single-class behaviour**

In [16]:

# PHASE 4E.3 — Single-class regional edge-case test
# Tests the existing evaluate_model() without changing it.

# Six regions, two observations per region.
# Region 1 contains only class 0.
# Regions 2–6 contain both classes.

y_edge = np.array([
    0, 0,       # North Central: single class
    0, 1,       # North East
    0, 1,       # North West
    0, 1,       # South East
    0, 1,       # South South
    0, 1        # South West
], dtype=np.float32)

regions_edge = np.repeat(np.arange(1, 7), 2)

probabilities_edge = np.array([
    0.1, 0.2,
    0.1, 0.8,
    0.2, 0.9,
    0.1, 0.9,
    0.4, 0.7,
    0.2, 0.8
])

logits_edge = np.log(
    probabilities_edge / (1 - probabilities_edge)
)

X_edge = np.zeros((len(y_edge), 1), dtype=np.float32)

model_edge = ControlledLogitModel(logits_edge)

print("RUNNING SINGLE-CLASS REGIONAL TEST")

try:
    edge_results = evaluate_model(
        model_edge,
        X_edge,
        y_edge,
        regions_edge
    )

    for result in edge_results:
        print(result)

    print("\nSINGLE-CLASS REGIONAL TEST: FUNCTION COMPLETED")

except Exception as exc:
    print("\nSINGLE-CLASS REGIONAL TEST: FUNCTION FAILED")
    print("Exception type:", type(exc).__name__)
    print("Exception message:", str(exc))
    raise


RUNNING SINGLE-CLASS REGIONAL TEST
{'region_id': 0, 'region': 'Pooled', 'n': 12, 'validation_loss': 0.2091684192419052, 'accuracy': 1.0}
{'region_id': 1, 'region': 'North Central', 'n': 2, 'validation_loss': 0.16425204277038574, 'accuracy': 1.0}
{'region_id': 2, 'region': 'North East', 'n': 2, 'validation_loss': 0.16425202786922455, 'accuracy': 1.0}
{'region_id': 3, 'region': 'North West', 'n': 2, 'validation_loss': 0.16425204277038574, 'accuracy': 1.0}
{'region_id': 4, 'region': 'South East', 'n': 2, 'validation_loss': 0.10536050796508789, 'accuracy': 1.0}
{'region_id': 5, 'region': 'South South', 'n': 2, 'validation_loss': 0.4337502717971802, 'accuracy': 1.0}
{'region_id': 6, 'region': 'South West', 'n': 2, 'validation_loss': 0.2231435626745224, 'accuracy': 1.0}

SINGLE-CLASS REGIONAL TEST: FUNCTION COMPLETED


## Phase 4 integration review

**Integration smoke passed: 4/4 runs**

All four combinations completed:

| Feature configuration | Algorithm | Inputs | Validation accuracy | Macro-F1 |
|---|---|---|---|---|
| Baseline | FedAvg | 15 | 0.5808 | 0.3705 |
| Baseline | FedProx | 15 | 0.5801 | 0.3687 |
| Expanded | FedAvg | 51 | 0.5799 | 0.3670 |
| Expanded | FedProx | 51 | 0.5799 | 0.3670 |

> These are one-round, one-local-epoch integration results, not final model-performance estimates. They establish that the four training/evaluation paths execute; they do not establish that either algorithm performs better.

## Phase 4 completion assessment

I also checked the v1.1 source for Python syntax and reviewed its training, evaluation-integration and output-saving paths.

| Requirement | Evidence | Status |
|---|---|---|
| Canonical FL implementation | v1.1 source reviewed | Passed |
| Reproducibility and sanity checks | v1.0 DET-01, DET-02, DET-03 passed | Passed |
| FedProx smoke checks | v1.0 checks passed | Passed |
| Evaluation metric validation | 29/29 controlled tests and 13/13 independent audit checks passed | Passed |
| v1.1 integration | Four combinations completed | Passed |
| Validation ID and output preservation | Confirmed in supplied smoke summary | Passed |
| Test partition protection | v1.1 loader and evaluator restrict access to train/validation | Passed |

## Phase 4 complete

I consider Phase 4 closed. The smoke-test results and earlier validation results should be retained as the evidence supporting this sign-off. No need to repeat those tests.

## Computattional feasibility and configuration-selection phase ( not confirmatory training)